In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

In [5]:
import sys
from pathlib import Path


BASE_DIR = Path().resolve().parent
sys.path.append(str(BASE_DIR / "src"))

In [ ]:
from preprocessing import *
from models import logistic_pipeline,logistic_gridsearch, randomforest_model, randomforest_gridsearch, xgboost_model, xgboost_gridsearch
from evaluate import run_cross_validation, evaluate_model

In [7]:
dff = pd.read_csv("../data/Telco_customer_churn.csv")

In [8]:
df = dff.drop(columns=["customerID", "gender", "PhoneService"])
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna()

In [9]:
df = feature_engineering(df)

In [10]:
X = df.drop("Churn", axis=1)
y = df["Churn"]
y = y.map({"Yes": 1, "No": 0})
X = pd.get_dummies(X, drop_first=True)
X = X.astype("float64")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [15]:
pipeline = logistic_pipeline()

cv_results = run_cross_validation(pipeline, X_train, y_train, cv=5)

print("=== Cross-validation results ===")
for metric_name, values in cv_results.items():
    if metric_name.startswith("test_"):
        print(f"{metric_name}: {values.mean():.4f}")

grid = logistic_gridsearch(cv=5, scoring="recall", n_jobs=-1)
grid.fit(X_train, y_train)

print("\n=== Best parameters ===")
print(grid.best_params_)

print("\n=== Best CV score ===")
print(f"{grid.best_score_:.4f}")

final_results = evaluate_model(grid.best_estimator_, X_test, y_test)

print("\n=== Test results ===")
print(f"Accuracy : {final_results['accuracy']:.4f}")
print(f"Precision : {final_results['precision']:.4f}")
print(f"Recall : {final_results['recall']:.4f}")
print(f"F1-score : {final_results['f1_score']:.4f}")

print("\n=== Classification report ===")
print(final_results["classification_report"])

=== Cross-validation results ===
test_accuracy: 0.8018
test_precision: 0.6578
test_recall: 0.5311
test_f1: 0.5875
test_roc_auc: 0.8469

=== Best parameters ===
{'logisticregression__C': 0.01, 'logisticregression__class_weight': 'balanced', 'logisticregression__solver': 'liblinear'}

=== Best CV score ===
0.8020

=== Test results ===
Accuracy : 0.7264
Precision : 0.4908
Recall : 0.7888
F1-score : 0.6051

=== Classification report ===
              precision    recall  f1-score   support

           0       0.90      0.70      0.79      1033
           1       0.49      0.79      0.61       374

    accuracy                           0.73      1407
   macro avg       0.70      0.75      0.70      1407
weighted avg       0.79      0.73      0.74      1407



In [16]:
model = randomforest_model()

cv_results = run_cross_validation(model, X_train, y_train, cv=5)

print("=== Cross-validation results ===")
for metric_name, values in cv_results.items():
    if metric_name.startswith("test_"):
        print(f"{metric_name}: {values.mean():.4f}")

grid = randomforest_gridsearch(cv=5, scoring="recall", n_jobs=-1)
grid.fit(X_train, y_train)

print("\n=== Best parameters ===")
print(grid.best_params_)

print("\n=== Best CV score ===")
print(f"{grid.best_score_:.4f}")

final_results = evaluate_model(grid.best_estimator_, X_test, y_test)

print("\n=== Test results ===")
print(f"Accuracy : {final_results['accuracy']:.4f}")
print(f"Precision : {final_results['precision']:.4f}")
print(f"Recall : {final_results['recall']:.4f}")
print(f"F1-score : {final_results['f1_score']:.4f}")

print("\n=== Classification report ===")
print(final_results["classification_report"])

=== Cross-validation results ===
test_accuracy: 0.7877
test_precision: 0.6306
test_recall: 0.4856
test_f1: 0.5486
test_roc_auc: 0.8226

=== Best parameters ===
{'class_weight': 'balanced', 'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}

=== Best CV score ===
0.7960

=== Test results ===
Accuracy : 0.7328
Precision : 0.4984
Recall : 0.8102
F1-score : 0.6171

=== Classification report ===
              precision    recall  f1-score   support

           0       0.91      0.70      0.79      1033
           1       0.50      0.81      0.62       374

    accuracy                           0.73      1407
   macro avg       0.70      0.76      0.71      1407
weighted avg       0.80      0.73      0.75      1407



In [17]:
model = xgboost_model()

cv_results = run_cross_validation(model, X_train, y_train, cv=5)

print("=== Cross-validation results ===")
for metric_name, values in cv_results.items():
    if metric_name.startswith("test_"):
        print(f"{metric_name}: {values.mean():.4f}")

grid = xgboost_gridsearch(cv=5, scoring="recall", n_jobs=-1)
grid.fit(X_train, y_train)

print("\n=== Best parameters ===")
print(grid.best_params_)

print("\n=== Best CV score ===")
print(f"{grid.best_score_:.4f}")

final_results = evaluate_model(grid.best_estimator_, X_test, y_test)

print("\n=== Test results ===")
print(f"Accuracy : {final_results['accuracy']:.4f}")
print(f"Precision : {final_results['precision']:.4f}")
print(f"Recall : {final_results['recall']:.4f}")
print(f"F1-score : {final_results['f1_score']:.4f}")

print("\n=== Classification report ===")
print(final_results["classification_report"])

=== Cross-validation results ===
test_accuracy: 0.7765
test_precision: 0.5932
test_recall: 0.5084
test_f1: 0.5474
test_roc_auc: 0.8207

=== Best parameters ===
{'colsample_bytree': 1.0, 'gamma': 0, 'learning_rate': 0.05, 'max_depth': 3, 'min_child_weight': 1, 'n_estimators': 100, 'scale_pos_weight': 3, 'subsample': 1.0}

=== Best CV score ===
0.8274

=== Test results ===
Accuracy : 0.7186
Precision : 0.4829
Recall : 0.8289
F1-score : 0.6102

=== Classification report ===
              precision    recall  f1-score   support

           0       0.92      0.68      0.78      1033
           1       0.48      0.83      0.61       374

    accuracy                           0.72      1407
   macro avg       0.70      0.75      0.69      1407
weighted avg       0.80      0.72      0.73      1407



# Pas d'amélioration nette. Le modèle baseline suffisait à prendre en compte toutes les données.